<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال دو ارزیابی مستقل داریم:
- تابع اول با استفاده از relevant_doc_ids، کیفیت Retrieval را با Precision و Recall می‌سنجد.
- تابع دوم با Qwen3 بررسی می‌کند آیا پاسخ تولیدشده توسط Context واقعی پشتیبانی می‌شود یا خیر.

در این مثال، انتظار داریم:
- Retrieval Precision پایین باشد؛ چون یکی از دو سند بازیابی‌شده نامرتبط است.
- Retrieval Recall برابر 0.5 باشد؛ چون از دو سند مورد انتظار فقط یکی پیدا شده است.
- Faithfulness پایین باشد؛ چون Context می‌گوید ۲۶ روز، اما مدل ۳۰ روز پاسخ داده است.

In [ ]:

import json
from openai import OpenAI

# 1. Connect to Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL_ID = "qwen3:4b"

# 2. Evaluation data
question = "How many annual leave days do employees receive?"

expected_doc_ids = {"HR-001", "HR-002"}

retrieved_docs = {
    "HR-001": "Employees receive 26 days of annual leave per year.",
    "HR-003": "Employees are covered by supplementary health insurance."
}

generated_answer = (
    "Employees receive 30 days of annual leave per year."
)

# 3. Evaluate Retrieval
def evaluate_retrieval(expected, retrieved):

    retrieved_ids = set(retrieved.keys())

    true_positives = expected & retrieved_ids

    precision = (
        len(true_positives) / len(retrieved_ids)
        if retrieved_ids else 0
    )

    recall = (
        len(true_positives) / len(expected)
        if expected else 0
    )

    return {
        "precision": precision,
        "recall": recall
    }


retrieval_result = evaluate_retrieval(
    expected_doc_ids,
    retrieved_docs
)

# 4. Evaluate Generation
def evaluate_generation(question, contexts, answer):

    prompt = f"""
Evaluate if the answer is supported by the context.

Question: {question}
Context: {json.dumps(contexts)}
Answer: {answer}

Score:
0 = Not supported
1 = Partially supported
2 = Fully supported

Return JSON only:
{{"score": 0, "reason": "Short explanation"}}
"""

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        extra_body={"think": False}
    )

    result = json.loads(
        response.choices[0].message.content
    )

    score = result["score"]

    if type(score) is not int or score not in (0, 1, 2):
        raise ValueError("Invalid evaluation score.")

    return {
        "faithfulness": score / 2,
        "reason": result["reason"]
    }


generation_result = evaluate_generation(
    question,
    list(retrieved_docs.values()),
    generated_answer
)

# 5. Display independent results
print("===== RETRIEVAL EVALUATION =====")
print(f"Precision: {retrieval_result['precision']:.2f}")
print(f"Recall:    {retrieval_result['recall']:.2f}")

print("\n===== GENERATION EVALUATION =====")
print(f"Faithfulness: {generation_result['faithfulness']:.2f}")
print("Reason:", generation_result["reason"])
